# Fase 5 - Detección de Anomalías en Demanda de Taxis NYC

**Pregunta de negocio:** ¿Qué días son anómalos en la demanda de taxis?

## Objetivos de este notebook

1. Identificar días con demanda inusualmente alta o baja usando tres métodos:
   - **Z-score:** método estadístico paramétrico
   - **IQR:** método robusto no paramétrico
   - **Isolation Forest:** método basado en machine learning
2. Correlacionar anomalías con eventos reales (tormentas, feriados)
3. Evaluar severidad de cada anomalía
4. Comparar la concordancia entre métodos

## ¿Por qué detectar anomalías en la demanda?

Identificar días atípicos tiene valor operativo concreto:
- **Planificación de contingencias:** prepararse para eventos que alteran la demanda
- **Pricing dinámico:** ajustar tarifas ante caídas o picos inesperados
- **Limpieza de datos:** excluir anomalías al entrenar modelos predictivos
- **Post-mortem operativo:** entender qué causó caídas de servicio

## 1. Configuración e importaciones

In [ ]:
# Conexión a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerías de análisis y visualización
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Para guardar resultados
import os

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Carga de datos diarios

Obtenemos el conteo diario de viajes y la tarifa promedio diaria para 2015.
Ambas variables nos dan perspectivas complementarias: volumen y valor monetario.

In [ ]:
query = """
SELECT
    DATE(pickup_datetime) AS fecha,
    COUNT(*) AS trip_count,
    AVG(fare_amount) AS avg_fare,
    AVG(trip_distance) AS avg_distance,
    AVG(tip_amount) AS avg_tip,
    SUM(total_amount) AS total_revenue
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    pickup_datetime BETWEEN '2015-01-01' AND '2015-12-31'
    AND fare_amount > 0
    AND fare_amount < 500
    AND trip_distance > 0
GROUP BY
    fecha
ORDER BY
    fecha
"""

# Cache para evitar consultas repetidas
cache_path = '../../../data/processed/nyc_taxi_daily_anomaly_2015.csv'
os.makedirs(os.path.dirname(cache_path), exist_ok=True)

if os.path.exists(cache_path):
    df = pd.read_csv(cache_path, parse_dates=['fecha'])
    print(f"Datos cargados desde cache: {len(df)} días")
else:
    df = bq.query_to_df(query)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df.to_csv(cache_path, index=False)
    print(f"Datos descargados de BigQuery y guardados: {len(df)} días")

df = df.set_index('fecha').sort_index()

# Agregar features temporales
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['day_name'] = df.index.day_name()

print(f"\nRango: {df.index.min().date()} a {df.index.max().date()}")
print(f"\nEstadísticas de viajes diarios:")
print(df['trip_count'].describe().apply(lambda x: f'{x:,.0f}'))

In [ ]:
# Visualización de la serie temporal
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.8)
axes[0].set_title('Viajes Diarios - NYC Taxi Amarillo 2015', fontweight='bold')
axes[0].set_ylabel('Número de viajes')
axes[0].axhline(df['trip_count'].mean(), color='red', linestyle='--', alpha=0.7,
                label=f"Media: {df['trip_count'].mean():,.0f}")
axes[0].legend()

axes[1].plot(df.index, df['avg_fare'], color='#4CAF50', linewidth=0.8)
axes[1].set_title('Tarifa Promedio Diaria', fontweight='bold')
axes[1].set_ylabel('Tarifa promedio (USD)')
axes[1].set_xlabel('Fecha')

plt.tight_layout()
plt.show()

## 3. Método 1: Detección por Z-score

El **z-score** mide cuántas desviaciones estándar se aleja un valor de la media:

$$z = \frac{x - \mu}{\sigma}$$

Usamos un umbral de |z| > 2.5, lo que significa que un día es anómalo si su demanda
está a más de 2.5 desviaciones estándar de la media.

**Supuesto:** asume distribución aproximadamente normal (puede fallar con datos sesgados).

In [ ]:
# Calcular z-scores para viajes diarios
mean_trips = df['trip_count'].mean()
std_trips = df['trip_count'].std()
df['zscore'] = (df['trip_count'] - mean_trips) / std_trips

# Umbral de anomalía
z_threshold = 2.5
df['anomaly_zscore'] = np.abs(df['zscore']) > z_threshold

# Clasificar tipo de anomalía
df['anomaly_zscore_type'] = 'Normal'
df.loc[df['zscore'] > z_threshold, 'anomaly_zscore_type'] = 'Pico (alta demanda)'
df.loc[df['zscore'] < -z_threshold, 'anomaly_zscore_type'] = 'Valle (baja demanda)'

# Resultados
n_anomalies_z = df['anomaly_zscore'].sum()
print(f"Anomalías detectadas por Z-score (|z| > {z_threshold}): {n_anomalies_z}")
print(f"  - Picos (alta demanda):  {(df['anomaly_zscore_type'] == 'Pico (alta demanda)').sum()}")
print(f"  - Valles (baja demanda): {(df['anomaly_zscore_type'] == 'Valle (baja demanda)').sum()}")

# Mostrar días anómalos
anomalies_z = df[df['anomaly_zscore']].sort_values('zscore')
print(f"\nDías anómalos (ordenados por z-score):")
print(anomalies_z[['trip_count', 'zscore', 'day_name', 'anomaly_zscore_type']].to_string())

In [ ]:
# Visualización: serie temporal con anomalías Z-score
fig, ax = plt.subplots(figsize=(18, 6))

# Serie normal
normal = df[~df['anomaly_zscore']]
anomalous = df[df['anomaly_zscore']]

ax.plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.7, alpha=0.7, label='Normal')

# Anomalías: valles en rojo, picos en naranja
valleys = anomalous[anomalous['zscore'] < 0]
peaks = anomalous[anomalous['zscore'] > 0]

ax.scatter(valleys.index, valleys['trip_count'], color='red', s=80, zorder=5,
           edgecolors='darkred', linewidth=1.5, label=f'Valles ({len(valleys)})')
ax.scatter(peaks.index, peaks['trip_count'], color='#FF9800', s=80, zorder=5,
           edgecolors='darkorange', linewidth=1.5, label=f'Picos ({len(peaks)})')

# Bandas de umbral
upper_z = mean_trips + z_threshold * std_trips
lower_z = mean_trips - z_threshold * std_trips
ax.axhline(upper_z, color='red', linestyle=':', alpha=0.5, label=f'Umbral z={z_threshold}')
ax.axhline(lower_z, color='red', linestyle=':', alpha=0.5)
ax.axhline(mean_trips, color='gray', linestyle='--', alpha=0.5, label='Media')

# Anotar algunos puntos clave
for idx, row in valleys.head(3).iterrows():
    ax.annotate(idx.strftime('%d-%b'), xy=(idx, row['trip_count']),
                xytext=(10, -25), textcoords='offset points', fontsize=8,
                arrowprops=dict(arrowstyle='->', color='red', lw=0.8))

ax.set_title('Detección de Anomalías por Z-score (|z| > 2.5)', fontweight='bold', fontsize=14)
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

## 4. Método 2: Detección por IQR

El **rango intercuartílico (IQR)** es un método más robusto que el z-score:

- IQR = Q3 - Q1
- Outlier inferior: x < Q1 - 1.5 * IQR
- Outlier superior: x > Q3 + 1.5 * IQR

**Ventaja:** No asume normalidad. Es resistente a valores extremos
porque se basa en percentiles en lugar de media/desviación estándar.

In [ ]:
# Calcular límites IQR
Q1 = df['trip_count'].quantile(0.25)
Q3 = df['trip_count'].quantile(0.75)
IQR = Q3 - Q1

lower_iqr = Q1 - 1.5 * IQR
upper_iqr = Q3 + 1.5 * IQR

df['anomaly_iqr'] = (df['trip_count'] < lower_iqr) | (df['trip_count'] > upper_iqr)

# Clasificar tipo
df['anomaly_iqr_type'] = 'Normal'
df.loc[df['trip_count'] > upper_iqr, 'anomaly_iqr_type'] = 'Pico'
df.loc[df['trip_count'] < lower_iqr, 'anomaly_iqr_type'] = 'Valle'

# Resultados
n_anomalies_iqr = df['anomaly_iqr'].sum()
print(f"Parámetros IQR:")
print(f"  Q1:             {Q1:,.0f}")
print(f"  Q3:             {Q3:,.0f}")
print(f"  IQR:            {IQR:,.0f}")
print(f"  Límite inferior: {lower_iqr:,.0f}")
print(f"  Límite superior: {upper_iqr:,.0f}")
print(f"\nAnomalías detectadas por IQR: {n_anomalies_iqr}")
print(f"  - Picos:  {(df['anomaly_iqr_type'] == 'Pico').sum()}")
print(f"  - Valles: {(df['anomaly_iqr_type'] == 'Valle').sum()}")

# Mostrar días anómalos
anomalies_iqr = df[df['anomaly_iqr']].sort_values('trip_count')
print(f"\nDías anómalos por IQR:")
print(anomalies_iqr[['trip_count', 'day_name', 'anomaly_iqr_type']].to_string())

In [ ]:
# Visualización: serie temporal con anomalías IQR
fig, ax = plt.subplots(figsize=(18, 6))

ax.plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.7, alpha=0.7)

anomalous_iqr = df[df['anomaly_iqr']]
valleys_iqr = anomalous_iqr[anomalous_iqr['trip_count'] < lower_iqr]
peaks_iqr = anomalous_iqr[anomalous_iqr['trip_count'] > upper_iqr]

ax.scatter(valleys_iqr.index, valleys_iqr['trip_count'], color='red', s=80, zorder=5,
           edgecolors='darkred', linewidth=1.5, label=f'Valles ({len(valleys_iqr)})')
ax.scatter(peaks_iqr.index, peaks_iqr['trip_count'], color='#FF9800', s=80, zorder=5,
           edgecolors='darkorange', linewidth=1.5, label=f'Picos ({len(peaks_iqr)})')

# Bandas IQR
ax.axhspan(lower_iqr, upper_iqr, alpha=0.08, color='green', label='Rango IQR normal')
ax.axhline(upper_iqr, color='red', linestyle=':', alpha=0.5)
ax.axhline(lower_iqr, color='red', linestyle=':', alpha=0.5)

ax.set_title('Detección de Anomalías por Método IQR', fontweight='bold', fontsize=14)
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

# Comparación rápida con Z-score
both = (df['anomaly_zscore'] & df['anomaly_iqr']).sum()
only_z = (df['anomaly_zscore'] & ~df['anomaly_iqr']).sum()
only_iqr = (~df['anomaly_zscore'] & df['anomaly_iqr']).sum()
print(f"\nComparación Z-score vs IQR:")
print(f"  Detectadas por ambos: {both}")
print(f"  Solo Z-score:         {only_z}")
print(f"  Solo IQR:             {only_iqr}")

## 5. Método 3: Isolation Forest

**Isolation Forest** es un algoritmo de ML no supervisado que detecta anomalías
"aislando" observaciones. La intuición es que los puntos anómalos son más fáciles
de separar del resto porque son pocos y diferentes.

**Ventajas sobre métodos estadísticos:**
- Puede detectar anomalías en espacios multivariados
- No asume ninguna distribución particular
- Escala bien a datasets grandes

Usamos múltiples features: trip_count, avg_fare, day_of_week, month.

In [ ]:
# Preparar features para Isolation Forest
features_if = ['trip_count', 'avg_fare', 'day_of_week', 'month']
X = df[features_if].copy()

# Escalar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajustar Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # Esperamos ~5% de anomalías
    random_state=42,
    n_estimators=200,
    max_samples='auto'
)
df['anomaly_if'] = iso_forest.fit_predict(X_scaled)
df['anomaly_if'] = df['anomaly_if'] == -1  # -1 = anomalía, 1 = normal

# Score de anomalía (más negativo = más anómalo)
df['anomaly_score'] = iso_forest.decision_function(X_scaled)

# Resultados
n_anomalies_if = df['anomaly_if'].sum()
print(f"Anomalías detectadas por Isolation Forest: {n_anomalies_if}")
print(f"Tasa de contaminación efectiva: {n_anomalies_if / len(df) * 100:.1f}%")

# Mostrar anomalías ordenadas por score
anomalies_if = df[df['anomaly_if']].sort_values('anomaly_score')
print(f"\nDías anómalos (ordenados por severidad):")
print(anomalies_if[['trip_count', 'avg_fare', 'day_name', 'anomaly_score']].to_string())

In [ ]:
# Visualización: Isolation Forest
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1: Serie temporal con anomalías
normal_if = df[~df['anomaly_if']]
anom_if = df[df['anomaly_if']]

axes[0].plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.7, alpha=0.7)
axes[0].scatter(anom_if.index, anom_if['trip_count'], color='red', s=80, zorder=5,
               edgecolors='darkred', linewidth=1.5, label=f'Anomalías ({len(anom_if)})')
axes[0].set_title('Anomalías Isolation Forest - Serie Temporal', fontweight='bold')
axes[0].set_ylabel('Viajes diarios')
axes[0].set_xlabel('Fecha')
axes[0].legend()
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b'))

# Panel 2: Scatter trip_count vs avg_fare coloreado por anomalía
axes[1].scatter(normal_if['trip_count'], normal_if['avg_fare'], 
               color='#2196F3', alpha=0.4, s=30, label='Normal')
axes[1].scatter(anom_if['trip_count'], anom_if['avg_fare'], 
               color='red', s=80, edgecolors='darkred', linewidth=1.5,
               label='Anomalía', zorder=5)
axes[1].set_title('Anomalías en Espacio Bidimensional', fontweight='bold')
axes[1].set_xlabel('Viajes diarios')
axes[1].set_ylabel('Tarifa promedio (USD)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Correlación con eventos reales

Para validar nuestros detectores de anomalías, cruzamos los días detectados
con eventos reales conocidos que ocurrieron en NYC durante 2015.

Eventos clave:
- **26-27 enero:** Blizzard Juno (tormenta de nieve masiva, prohibición de circular)
- **4 julio:** Día de la Independencia
- **26 noviembre:** Acción de Gracias (Thanksgiving)
- **25 diciembre:** Navidad
- **31 diciembre:** Nochevieja

In [ ]:
# Definir eventos conocidos
events = pd.DataFrame([
    {'fecha': '2015-01-01', 'evento': 'Año Nuevo', 'tipo': 'Feriado'},
    {'fecha': '2015-01-19', 'evento': 'Martin Luther King Day', 'tipo': 'Feriado'},
    {'fecha': '2015-01-26', 'evento': 'Blizzard Juno (inicio)', 'tipo': 'Clima extremo'},
    {'fecha': '2015-01-27', 'evento': 'Blizzard Juno (pico)', 'tipo': 'Clima extremo'},
    {'fecha': '2015-02-14', 'evento': 'San Valentín', 'tipo': 'Evento social'},
    {'fecha': '2015-02-16', 'evento': 'Presidents Day', 'tipo': 'Feriado'},
    {'fecha': '2015-05-25', 'evento': 'Memorial Day', 'tipo': 'Feriado'},
    {'fecha': '2015-07-04', 'evento': 'Día de la Independencia', 'tipo': 'Feriado'},
    {'fecha': '2015-09-07', 'evento': 'Labor Day', 'tipo': 'Feriado'},
    {'fecha': '2015-11-26', 'evento': 'Thanksgiving', 'tipo': 'Feriado'},
    {'fecha': '2015-11-27', 'evento': 'Black Friday', 'tipo': 'Evento comercial'},
    {'fecha': '2015-12-25', 'evento': 'Navidad', 'tipo': 'Feriado'},
    {'fecha': '2015-12-31', 'evento': 'Nochevieja', 'tipo': 'Feriado'},
])
events['fecha'] = pd.to_datetime(events['fecha'])
events = events.set_index('fecha')

# Merge con datos diarios
df['evento'] = events['evento']
df['tipo_evento'] = events['tipo']
df['has_event'] = df['evento'].notna()

# Verificar cuántos eventos fueron detectados por cada método
event_days = df[df['has_event']].copy()

print("DETECCIÓN DE EVENTOS CONOCIDOS POR CADA MÉTODO")
print("=" * 75)
print(f"{'Fecha':<12} {'Evento':<25} {'Z-score':<10} {'IQR':<10} {'Iso.Forest':<10} {'Viajes':>12}")
print("-" * 75)
for idx, row in event_days.iterrows():
    z = 'SI' if row['anomaly_zscore'] else 'no'
    iqr = 'SI' if row['anomaly_iqr'] else 'no'
    iso = 'SI' if row['anomaly_if'] else 'no'
    print(f"{idx.strftime('%Y-%m-%d'):<12} {row['evento']:<25} {z:<10} {iqr:<10} {iso:<10} {row['trip_count']:>12,.0f}")

# Tasa de detección
total_events = len(event_days)
print(f"\nTasa de detección:")
print(f"  Z-score:         {event_days['anomaly_zscore'].sum()}/{total_events} ({event_days['anomaly_zscore'].mean()*100:.0f}%)")
print(f"  IQR:             {event_days['anomaly_iqr'].sum()}/{total_events} ({event_days['anomaly_iqr'].mean()*100:.0f}%)")
print(f"  Isolation Forest: {event_days['anomaly_if'].sum()}/{total_events} ({event_days['anomaly_if'].mean()*100:.0f}%)")

## 7. Ranking de severidad de anomalías

No todas las anomalías son iguales. Creamos un ranking de severidad basado en:
- El z-score absoluto (cuánto se desvía de la media)
- El score del Isolation Forest (cuán aislado está el punto)
- La coincidencia entre métodos (más métodos lo detectan = más confiable)

In [ ]:
# Crear score de severidad compuesto
# Combinar los tres métodos
df['n_methods_flagged'] = (
    df['anomaly_zscore'].astype(int) + 
    df['anomaly_iqr'].astype(int) + 
    df['anomaly_if'].astype(int)
)

# Cualquier día detectado por al menos un método
df['is_any_anomaly'] = df['n_methods_flagged'] > 0

# Score de severidad: normalizar z-score e isolation score, luego promediar
df['zscore_norm'] = np.abs(df['zscore']) / np.abs(df['zscore']).max()
df['if_score_norm'] = 1 - (df['anomaly_score'] - df['anomaly_score'].min()) / (
    df['anomaly_score'].max() - df['anomaly_score'].min()
)
df['severity'] = (df['zscore_norm'] + df['if_score_norm'] + df['n_methods_flagged'] / 3) / 3

# Ranking de las anomalías más severas
all_anomalies = df[df['is_any_anomaly']].sort_values('severity', ascending=False)

print("TOP 15 ANOMALÍAS MÁS SEVERAS")
print("=" * 85)
ranking = all_anomalies.head(15)[['trip_count', 'zscore', 'anomaly_score', 
                                   'n_methods_flagged', 'severity', 'day_name', 'evento']].copy()
ranking['rank'] = range(1, len(ranking) + 1)
ranking = ranking[['rank', 'trip_count', 'zscore', 'n_methods_flagged', 'severity', 'day_name', 'evento']]
ranking.columns = ['Rank', 'Viajes', 'Z-score', 'Métodos', 'Severidad', 'Día', 'Evento']
print(ranking.to_string())

In [ ]:
# Visualización del ranking de severidad
fig, ax = plt.subplots(figsize=(18, 6))

# Colorear por severidad
scatter = ax.scatter(df.index, df['trip_count'], c=df['severity'], 
                     cmap='YlOrRd', s=15, alpha=0.6)

# Resaltar top 10 anomalías
top10 = all_anomalies.head(10)
ax.scatter(top10.index, top10['trip_count'], color='red', s=120, 
           edgecolors='black', linewidth=2, zorder=5, label='Top 10 anomalías')

# Anotar top 5
for i, (idx, row) in enumerate(top10.head(5).iterrows()):
    label = row['evento'] if pd.notna(row['evento']) else idx.strftime('%d-%b')
    ax.annotate(label, xy=(idx, row['trip_count']),
                xytext=(10, 15 if i % 2 == 0 else -25), textcoords='offset points',
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.colorbar(scatter, ax=ax, label='Score de Severidad', shrink=0.8)
ax.set_title('Mapa de Severidad de Anomalías - NYC Taxi 2015', fontweight='bold', fontsize=14)
ax.set_ylabel('Viajes diarios')
ax.set_xlabel('Fecha')
ax.legend(loc='upper right')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.tight_layout()
plt.show()

## 8. Análisis de falsos positivos

Las anomalías detectadas que **no corresponden a eventos conocidos** podrían ser:
- **Verdaderas anomalías** que no incluimos en nuestra lista de eventos
- **Falsos positivos** causados por variabilidad normal

Investigarlas puede revelar eventos que desconocíamos.

In [ ]:
# Anomalías sin evento conocido asociado
false_positives = df[(df['is_any_anomaly']) & (~df['has_event'])].copy()
false_positives = false_positives.sort_values('severity', ascending=False)

print(f"Anomalías sin evento conocido asociado: {len(false_positives)}")
print(f"\nEstas podrían ser eventos que no incluimos en nuestra lista,")
print(f"o variaciones normales que los métodos interpretan como anómalas.\n")

print(f"{'Fecha':<12} {'Día':<12} {'Viajes':>10} {'Z-score':>10} {'Métodos':>8} {'Severidad':>10}")
print("-" * 65)
for idx, row in false_positives.iterrows():
    print(f"{idx.strftime('%Y-%m-%d'):<12} {row['day_name']:<12} {row['trip_count']:>10,.0f} "
          f"{row['zscore']:>10.2f} {row['n_methods_flagged']:>8} {row['severity']:>10.3f}")

print(f"\nPatrones en falsos positivos:")
print(f"  Por día de la semana:")
print(false_positives['day_name'].value_counts().to_string())
print(f"\n  Por mes:")
print(false_positives['month'].value_counts().sort_index().to_string())

## 9. Comparación de métodos: tabla de concordancia

¿Cuánto coinciden los tres métodos? Un alto acuerdo entre métodos distintos
nos da mayor confianza en la detección.

In [ ]:
# Tabla de concordancia
print("TABLA DE CONCORDANCIA ENTRE MÉTODOS")
print("=" * 55)

# Contar combinaciones
concordance = pd.DataFrame({
    'Combinación': [
        'Detectado por los 3 métodos',
        'Solo Z-score + IQR',
        'Solo Z-score + Isolation Forest',
        'Solo IQR + Isolation Forest',
        'Solo Z-score',
        'Solo IQR',
        'Solo Isolation Forest',
        'No detectado (normal)'
    ],
    'Días': [
        ((df['anomaly_zscore']) & (df['anomaly_iqr']) & (df['anomaly_if'])).sum(),
        ((df['anomaly_zscore']) & (df['anomaly_iqr']) & (~df['anomaly_if'])).sum(),
        ((df['anomaly_zscore']) & (~df['anomaly_iqr']) & (df['anomaly_if'])).sum(),
        ((~df['anomaly_zscore']) & (df['anomaly_iqr']) & (df['anomaly_if'])).sum(),
        ((df['anomaly_zscore']) & (~df['anomaly_iqr']) & (~df['anomaly_if'])).sum(),
        ((~df['anomaly_zscore']) & (df['anomaly_iqr']) & (~df['anomaly_if'])).sum(),
        ((~df['anomaly_zscore']) & (~df['anomaly_iqr']) & (df['anomaly_if'])).sum(),
        ((~df['anomaly_zscore']) & (~df['anomaly_iqr']) & (~df['anomaly_if'])).sum(),
    ]
})
concordance['Porcentaje'] = (concordance['Días'] / len(df) * 100).round(1)
print(concordance.to_string(index=False))

# Resumen
total_any = df['is_any_anomaly'].sum()
total_all3 = ((df['anomaly_zscore']) & (df['anomaly_iqr']) & (df['anomaly_if'])).sum()
print(f"\nResumen:")
print(f"  Total de días anómalos (al menos 1 método): {total_any}")
print(f"  Detectados por los 3 métodos: {total_all3} (alta confianza)")
print(f"  Tasa de acuerdo completo: {total_all3 / total_any * 100:.1f}% de las anomalías")

In [ ]:
# Visualización Venn-style usando barras apiladas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Distribución de concordancia
method_counts = df['n_methods_flagged'].value_counts().sort_index()
colors_bar = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']
labels_bar = ['Normal (0)', '1 método', '2 métodos', '3 métodos']
bars = axes[0].bar(method_counts.index, method_counts.values, color=colors_bar[:len(method_counts)])
axes[0].set_title('Distribución de Concordancia entre Métodos', fontweight='bold')
axes[0].set_xlabel('Número de métodos que detectan anomalía')
axes[0].set_ylabel('Número de días')
axes[0].set_xticks(method_counts.index)
axes[0].set_xticklabels(labels_bar[:len(method_counts)])
for bar, val in zip(bars, method_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', fontweight='bold')

# Panel 2: Heatmap de acuerdo pareado
methods = ['Z-score', 'IQR', 'Isolation Forest']
method_cols = ['anomaly_zscore', 'anomaly_iqr', 'anomaly_if']
agreement_matrix = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        agreement_matrix[i, j] = (df[method_cols[i]] & df[method_cols[j]]).sum()

sns.heatmap(agreement_matrix.astype(float), annot=True, fmt='.0f', cmap='YlOrRd',            xticklabels=methods, yticklabels=methods, ax=axes[1])
axes[1].set_title('Acuerdo Pareado entre Métodos', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Conclusiones y valor de negocio

### Hallazgos principales

1. **Blizzard Juno (26-27 enero)** fue la anomalía más severa del año: la demanda
   cayó drásticamente por la prohibición de circular durante la tormenta de nieve.

2. **Feriados importantes** (Navidad, Thanksgiving, 4 de julio) consistentemente
   muestran demanda anormal, generalmente hacia la baja.

3. **Nochevieja** puede mostrar un patrón mixto: demanda más baja durante el día
   pero posiblemente más alta en la noche (no visible en datos diarios).

### Comparación de métodos

| Método | Fortaleza | Debilidad | Recomendación |
|--------|-----------|-----------|---------------|
| Z-score | Simple, interpretable | Asume normalidad | Screening inicial |
| IQR | Robusto, no paramétrico | Solo univariado | Complemento del z-score |
| Isolation Forest | Multivariado, flexible | Menos interpretable | Detección sofisticada |

### Valor de negocio

- **Planificación proactiva:** Saber que feriados y eventos climáticos causan anomalías
  permite ajustar la flota con anticipación
- **Surge pricing informado:** Los días con demanda anormalmente alta son candidatos
  para precios dinámicos
- **Limpieza de modelos:** Excluir días anómalos mejora la precisión de los modelos
  de pronóstico (Notebook 04)
- **Alertas operativas:** Un sistema que detecte anomalías en tiempo real puede
  activar protocolos de contingencia

### Próximos pasos

- Notebook 06: Patrones de demanda por zona y hora para reposicionar la flota